## Merge Author Profiles — One-ORCID-Anchor, Size-2, No-Middle Names

Phase 4 of the author-merge initiative. Extends Phase 2's rich-name + signal recipe to the **no-middle (Tier-3, "Jason Priem" class)** parsed-name cohort, restricted to **size-2 groups with exactly one ORCID-bearing anchor profile**.

Companion to:
- `MergeAuthorsOrcidCollision` (Phase 1) — same-ORCID merges
- `MergeAuthorsRichName` (Phase 2) — size-2 rich-name + signal (excludes no-middle)
- `MergeAuthorsOrcidAnchorMulti` (Phase 3b) — size-3–9 one-ORCID-anchor rich-name
- `MergeAuthorsRichNameMulti` (Phase 3c) — size-3–9 both-no-ORCID rich-name

Phases 2 / 3b / 3c all require `p_middle` to contain a word ≥ 3 chars (rich-name guard). Phase 4 covers the residual **no-middle** parsed-name cohort that the rich-name guard explicitly excludes — names like "Jason Priem" parse to `('jason', '', 'priem')` and never qualify for Phases 2 / 3b / 3c.

**Scope:** size-2 groups, exactly one ORCID-bearing profile (the anchor), no-middle parsed names, with both inst overlap AND coauthor overlap ≥ 2 between the anchor and straggler. Audit (2026-05-02, 4 rounds n=24/52/52/100): 98% realistic precision, 89% conservative on n=100.

**Volume estimate:** ~103,287 candidate pairs.

**Precision guards (additive on Phase 3b/c recipe, with Phase-4-specific tightening):**

1. **First/last length ≥ 2 AFTER stripping trailing period.** `LENGTH(REGEXP_REPLACE(parsed.first, '[.]$', '')) >= 2` (same as Phase 3c). Critical for Phase 4 because, without a middle to anchor on, initials-first cases like "S. J. Arbiol Val" (parses as `('sj', '', 'arbiol val')`) would otherwise collapse multiple humans.
2. **No-middle filter.** Parsed `middle` must contain NO word ≥ 3 chars (the inverse of Phase 2/3b/3c's rich-name filter). Allows empty middle, initials-only middle, or punctuation-only middle.
3. **First-name vowel guard (new in Phase 4).** Parsed `first` (after dot-strip, lowercased) must contain at least one vowel (`a/e/i/o/u/y`). Catches initials-string cases like `"sj"`, `"dj"`, `"mj"` that pass the length-≥-2 check but are clearly initials. Loses ~66 pairs (negligible) but eliminates the entire initials-first failure class surfaced in v1 audit.
4. **Dominant raw_name majority (≥ 50%).** Same as Phases 2 / 3b / 3c.
5. **Year-in-full_name guard (new in Phase 4).** `full_name` must NOT contain any 4-digit run (e.g., "Baigent, Michael 1948-2013"). Librarian-curated metadata sometimes includes life dates in the name field; profiles with life dates are typically deceased authors whose work is incidentally co-cited with a same-name living author. v2 audit surfaced one such FP (Michael Baigent the writer vs. Michael F. Baigent the Australian psychiatrist).
6. **Middle-initial-from-dom_raw guard (new in Phase 4).** When `p_middle` is non-empty (initials-only, e.g., "t" or "B. D."), every profile's `full_name` (lowercased + ASCII-folded) must contain the first initial of `p_middle` as a single-letter token. Catches cases like "Michael T. Flannery" (dom_raw) vs anchor full_name "Michael P. Flannery" — different middle initials → different humans.
7. **Full_name implied-initial agreement guard (new in Phase 4).** For each profile, extract single-letter alphabetic tokens from the lowercased + ASCII-folded `full_name`. The pair fails if BOTH profiles have non-empty initial sets AND the sets don't intersect. Catches cases where `p_middle` is empty (no parsed initial to check via guard 6) but the full names disagree on middle initial — canonical example from v3 audit: anchor full_name "Ewa A. Kazimierska" vs straggler full_name "Ewa M. Kazimierska", same parsed key `('ewa', '', 'kazimierska')`, different middle initials in full names → likely two distinct humans (Ewa Anna vs Ewa Maria).
8. **Full_name self-consistency.** Every profile's `full_name` must contain `p_last` as a whole token (lowercased + ASCII-folded). Reduced from the Phase 3b version (which also checked the first ≥3-char middle word) because Phase 4's no-middle filter means there's no middle word to check.
9. **Group size = 2.** Size-3+ no-middle groups have unique false-positive risk (multiple distinct people sharing a common 2-token name + accumulating in the same name block); excluded.
10. **Sink cap.** `works_count <= 5,000` on every profile (same as Phases 1 / 2 / 3b / 3c).
11. **Exactly one ORCID-bearing profile per group.** Both the count of ORCID-bearing profiles AND the count of distinct ORCIDs equal 1.
12. **Per-pair precision signal (Option B from audit, stronger than Phase 2's recipe).** The pair must have `inst_overlap = TRUE` AND `n_shared_coauthors >= 2`, where coauthors exclude the other group member. Phase 2 used `inst_overlap OR n_shared_coauthors >= 1`; the no-middle cohort needs the stricter conjunction because, without rich-name signal, weak signal pairs collapse different humans (career-move ambiguity is the dominant failure mode at low signal strength).

**Winner selection:** the ORCID-bearing anchor wins, by construction.

**Loser handling:** *No DELETE.* Same MERGE-then-soft-zero-via-CreateAuthors approach as Phases 1 / 2 / 3b / 3c.

**Rollback plan:** No automatic rollback. Audit log `openalex.authors.author_merge_log_no_middle_anchor` captures the anchor's ORCID, parsed name, signal flags, and per-profile metadata; sufficient to reconstruct any loser via `STILL_UNMATCHED` + `MatchAuthors`.

**Known residual failure modes (out of scope for Phase 4):**
- **Different-careers same-parsed-name:** distinctive name appearing once at academic + once at industry, with COA = 2–4. Cannot be distinguished from a single person who career-changed without external data. Audit v4 surfaced ~9% of pairs in this shape, mostly clean but ~2 of 100 are likely FPs.
- **Coauthor-only no-INST pairs:** Phase 4's INST AND COA requirement excludes pairs that match only on coauthors. Notable example: Jason Priem's A5108365034 splinter (4 shared coauthors with the canonical, no inst overlap) — would need a separate Phase 4b with COA-only signal to catch.
- **Comma-form parser inversions:** "Jason, Priem" parses to `('priem', '', 'jason')` — different parsed key from "Jason Priem", so won't group with same-person profiles. Needs a CreateAuthorNames parser fix or unordered first/last grouping.


### Cell 1: Build merge candidates table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.merge_candidates_no_middle_anchor AS
WITH
all_profiles AS (
  SELECT id AS author_id, orcid, full_name, display_name, block_key,
         works_count, cited_by_count, created_date,
         TRANSFORM(COALESCE(last_known_institutions, ARRAY()), x -> x.id) AS lki_ids,
         TRANSFORM(COALESCE(affiliations,            ARRAY()), x -> x.institution.id) AS aff_ids
  FROM openalex.authors.openalex_authors
),
ra_counts AS (
  SELECT wa.author_id, wa.raw_author_name, COUNT(*) AS n_works
  FROM openalex.works.work_authors wa
  JOIN all_profiles n ON n.author_id = wa.author_id
  WHERE wa.raw_author_name IS NOT NULL AND TRIM(wa.raw_author_name) <> ''
  GROUP BY wa.author_id, wa.raw_author_name
),
ra_dominant AS (
  -- Dominant raw_name per profile, weighted by work_authors row count.
  -- Require >= 50% majority (Phase 2 recipe).
  SELECT author_id, raw_author_name
  FROM (
    SELECT author_id, raw_author_name, n_works,
           SUM(n_works) OVER (PARTITION BY author_id) AS total_works,
           ROW_NUMBER() OVER (PARTITION BY author_id
                              ORDER BY n_works DESC, raw_author_name ASC) AS rn
    FROM ra_counts
  )
  WHERE rn = 1 AND n_works * 2 >= total_works
),
parsed_dominant AS (
  -- Resolve dominant raw_name to (first, middle, last). Phase 4 filters:
  --  * first/last >= 2 chars AFTER stripping trailing period (initials-first defense)
  --  * middle has NO word >= 3 chars (no-middle cohort -- inverse of rich-name)
  --  * first contains a vowel (rejects "SJ", "DJ" all-initials cases)
  SELECT rd.author_id,
         REGEXP_REPLACE(an.parsed_name.first,  '[.]$', '') AS p_first,
         REGEXP_REPLACE(an.parsed_name.middle, '[.]$', '') AS p_middle,
         REGEXP_REPLACE(an.parsed_name.last,   '[.]$', '') AS p_last
  FROM ra_dominant rd
  JOIN openalex.authors.author_names an ON an.raw_author_name = rd.raw_author_name
  WHERE LENGTH(REGEXP_REPLACE(an.parsed_name.first, '[.]$', '')) >= 2
    AND LENGTH(REGEXP_REPLACE(an.parsed_name.last,  '[.]$', '')) >= 2
    AND SIZE(FILTER(
          SPLIT(REGEXP_REPLACE(COALESCE(an.parsed_name.middle,''), '[.]', ''), ' +'),
          x -> LENGTH(x) >= 3
        )) = 0
    AND REGEXP_LIKE(LOWER(REGEXP_REPLACE(an.parsed_name.first, '[.]', '')), '[aeiouy]')
),
profile_with_dom AS (
  SELECT n.*, p.p_first, p.p_middle, p.p_last,
         (n.orcid IS NOT NULL AND n.orcid <> '') AS has_orcid,
         -- First initial letter of p_middle (lowercased), for the middle-initial guard.
         -- Empty if p_middle is empty/whitespace.
         LOWER(SUBSTR(REGEXP_REPLACE(p.p_middle, '[. ]', ''), 1, 1)) AS p_middle_initial,
         -- Implied middle-initial set from full_name: distinct single-letter alphabetic
         -- tokens after lowercasing + ASCII-folding. Used by the fn_initials agreement guard.
         ARRAY_DISTINCT(FILTER(
           SPLIT(LOWER(REGEXP_REPLACE(TRANSLATE(COALESCE(n.full_name, ''),
             'áéíóúüñçÁÉÍÓÚÜÑÇàèìòùâêîôûÀÈÌÒÙÂÊÎÔÛãõÃÕıİğĞşŞçÇöÖüÜæøłÆØŁčćžšŠ',
             'aeiouuncAEIOUUNCaeiouaeiouAEIOUAEIOUaoaoiIgGsScCoOuUaolAOLccszS'),
             '[^a-zA-Z]', ' ')), ' +'),
           x -> LENGTH(x) = 1 AND x RLIKE '[a-z]'
         )) AS fn_initials
  FROM all_profiles    n
  JOIN parsed_dominant p ON p.author_id = n.author_id
),
profile_consistent AS (
  -- Per-profile guards:
  --  * full_name not null
  --  * year-in-full_name guard (no 4-digit runs -- catches life-dates pollution)
  --  * full_name self-consistency: must contain p_last as a whole token
  --  * middle-initial-from-dom_raw guard: when p_middle is non-empty,
  --    full_name must contain that initial as a single-letter token
  SELECT pwd.*,
    CASE
      WHEN full_name IS NULL THEN FALSE
      WHEN REGEXP_LIKE(full_name, '[0-9]{4}') THEN FALSE
      WHEN INSTR(
        CONCAT(' ', LOWER(REGEXP_REPLACE(
          TRANSLATE(full_name,
            'áéíóúüñçÁÉÍÓÚÜÑÇàèìòùâêîôûÀÈÌÒÙÂÊÎÔÛãõÃÕıİğĞşŞçÇöÖüÜæøłÆØŁčćžšŠ',
            'aeiouuncAEIOUUNCaeiouaeiouAEIOUAEIOUaoaoiIgGsScCoOuUaolAOLccszS'),
          '[^a-zA-Z]', ' ')), ' '),
        CONCAT(' ', p_last, ' ')
      ) = 0 THEN FALSE
      WHEN p_middle_initial <> ''
           AND INSTR(
             CONCAT(' ', LOWER(REGEXP_REPLACE(
               TRANSLATE(full_name,
                 'áéíóúüñçÁÉÍÓÚÜÑÇàèìòùâêîôûÀÈÌÒÙÂÊÎÔÛãõÃÕıİğĞşŞçÇöÖüÜæøłÆØŁčćžšŠ',
                 'aeiouuncAEIOUUNCaeiouaeiouAEIOUAEIOUaoaoiIgGsScCoOuUaolAOLccszS'),
               '[^a-zA-Z]', ' ')), ' '),
             CONCAT(' ', p_middle_initial, ' ')
           ) = 0
        THEN FALSE
      ELSE TRUE
    END AS full_name_ok
  FROM profile_with_dom pwd
),
qualifying_groups AS (
  -- Size = 2, exactly one ORCID-bearing profile, sink cap, all profiles full_name_ok.
  SELECT p_first, p_middle, p_last
  FROM profile_consistent
  GROUP BY p_first, p_middle, p_last
  HAVING COUNT(*) = 2
     AND MAX(works_count) <= 5000
     AND COUNT(CASE WHEN has_orcid THEN 1 END) = 1
     AND COUNT(DISTINCT CASE WHEN has_orcid THEN orcid END) = 1
     AND COUNT(*) = COUNT(CASE WHEN full_name_ok THEN 1 END)
),
group_profiles AS (
  SELECT pc.*
  FROM profile_consistent pc
  JOIN qualifying_groups g
    ON pc.p_first = g.p_first AND pc.p_middle = g.p_middle AND pc.p_last = g.p_last
),
anchor AS (
  SELECT p_first, p_middle, p_last,
         author_id AS anchor_id,
         orcid     AS anchor_orcid,
         ARRAY_DISTINCT(CONCAT(lki_ids, aff_ids)) AS anchor_inst,
         fn_initials AS anchor_fn_initials
  FROM group_profiles
  WHERE has_orcid
),
straggler AS (
  SELECT p_first, p_middle, p_last,
         author_id AS straggler_id,
         ARRAY_DISTINCT(CONCAT(lki_ids, aff_ids)) AS straggler_inst,
         fn_initials AS straggler_fn_initials
  FROM group_profiles
  WHERE NOT has_orcid
),
pair_inst AS (
  -- Compute inst_overlap and the fn_initials agreement guard at the pair level.
  SELECT a.p_first, a.p_middle, a.p_last,
         a.anchor_id, s.straggler_id,
         (SIZE(ARRAY_INTERSECT(a.anchor_inst, s.straggler_inst)) > 0) AS inst_overlap,
         -- fn_initials agreement: pass if either set is empty, or they overlap.
         (SIZE(a.anchor_fn_initials) = 0
          OR SIZE(s.straggler_fn_initials) = 0
          OR SIZE(ARRAY_INTERSECT(a.anchor_fn_initials, s.straggler_fn_initials)) > 0) AS fn_initials_ok
  FROM anchor a
  JOIN straggler s
    ON a.p_first = s.p_first AND a.p_middle = s.p_middle AND a.p_last = s.p_last
),
group_member_coauthors AS (
  -- Per profile, the set of distinct coauthor author_ids on its work_authors rows.
  SELECT wa1.author_id AS profile_id, wa2.author_id AS coauthor_id
  FROM openalex.works.work_authors wa1
  JOIN openalex.works.work_authors wa2
    ON wa2.work_id = wa1.work_id
   AND wa2.author_id <> wa1.author_id
  WHERE wa1.author_id IN (SELECT anchor_id    FROM anchor)
     OR wa1.author_id IN (SELECT straggler_id FROM straggler)
  GROUP BY wa1.author_id, wa2.author_id
),
pair_coauthor AS (
  -- Coauthors shared between anchor and straggler, EXCLUDING the other group member.
  SELECT p.p_first, p.p_middle, p.p_last,
         p.anchor_id, p.straggler_id,
         COUNT(DISTINCT ac.coauthor_id) AS n_shared_coauthors
  FROM pair_inst p
  JOIN group_member_coauthors ac ON ac.profile_id = p.anchor_id
  JOIN group_member_coauthors sc ON sc.profile_id = p.straggler_id
                                AND sc.coauthor_id = ac.coauthor_id
  WHERE ac.coauthor_id NOT IN (SELECT anchor_id    FROM anchor)
    AND ac.coauthor_id NOT IN (SELECT straggler_id FROM straggler)
  GROUP BY p.p_first, p.p_middle, p.p_last, p.anchor_id, p.straggler_id
),
pair_signal AS (
  SELECT pi.p_first, pi.p_middle, pi.p_last,
         pi.anchor_id, pi.straggler_id,
         pi.inst_overlap, pi.fn_initials_ok,
         COALESCE(pc.n_shared_coauthors, 0) AS n_shared_coauthors
  FROM pair_inst pi
  LEFT JOIN pair_coauthor pc
    ON pc.anchor_id = pi.anchor_id AND pc.straggler_id = pi.straggler_id
)
SELECT
  ps.p_first, ps.p_middle, ps.p_last,
  ps.anchor_id,
  a.anchor_orcid,
  ps.straggler_id,
  ps.inst_overlap,
  ps.n_shared_coauthors,
  -- Anchor metadata
  ap.full_name      AS anchor_full_name,
  ap.works_count    AS anchor_works_count,
  ap.cited_by_count AS anchor_cited_by_count,
  ap.created_date   AS anchor_created_date,
  ap.block_key      AS anchor_block_key,
  -- Straggler (loser) metadata
  sp.full_name      AS straggler_full_name,
  sp.orcid          AS straggler_orcid,
  sp.works_count    AS straggler_works_count,
  sp.cited_by_count AS straggler_cited_by_count,
  sp.created_date   AS straggler_created_date,
  sp.block_key      AS straggler_block_key
FROM pair_signal ps
JOIN anchor a
  ON a.p_first = ps.p_first AND a.p_middle = ps.p_middle
 AND a.p_last  = ps.p_last  AND a.anchor_id = ps.anchor_id
JOIN group_profiles ap ON ap.author_id = ps.anchor_id
JOIN group_profiles sp ON sp.author_id = ps.straggler_id
-- Phase 4 v4 signal floor: REQUIRE inst_overlap AND n_shared_coauthors >= 2,
-- AND fn_initials_ok (Guard 3 — full_name implied-initial agreement).
WHERE ps.inst_overlap = TRUE
  AND ps.n_shared_coauthors >= 2
  AND ps.fn_initials_ok = TRUE


### Cell 2: Validation statistics


In [ ]:
%sql
SELECT
  COUNT(DISTINCT anchor_id)                                                      AS n_groups,
  COUNT(*)                                                                       AS n_losers,
  COUNT(DISTINCT anchor_id, straggler_id)                                        AS n_pairs,
  SUM(straggler_works_count)                                                     AS works_to_rewrite,
  COUNT(CASE WHEN n_shared_coauthors >= 5 THEN 1 END)                            AS pairs_with_coa_5plus,
  COUNT(CASE WHEN n_shared_coauthors >= 10 THEN 1 END)                           AS pairs_with_coa_10plus,
  COUNT(CASE WHEN straggler_orcid IS NOT NULL AND straggler_orcid <> ''
              THEN 1 END)                                                        AS losers_with_orcid_unexpected,
  PERCENTILE_APPROX(n_shared_coauthors, ARRAY(0.5, 0.95, 0.99))                  AS coauthor_pctiles,
  PERCENTILE_APPROX(straggler_works_count, ARRAY(0.5, 0.95, 0.99))               AS straggler_works_pctiles
FROM openalex.authors.merge_candidates_no_middle_anchor


### Cell 3: Spot-check sample (manual review)


In [ ]:
%sql
WITH sampled_anchors AS (
  SELECT DISTINCT anchor_id
  FROM openalex.authors.merge_candidates_no_middle_anchor
  ORDER BY RAND() LIMIT 30
)
SELECT c.anchor_id, c.anchor_orcid, c.p_first, c.p_middle, c.p_last,
       c.anchor_full_name, c.anchor_works_count, c.anchor_cited_by_count,
       c.straggler_id, c.straggler_full_name,
       c.straggler_works_count, c.straggler_cited_by_count,
       c.inst_overlap, c.n_shared_coauthors
FROM openalex.authors.merge_candidates_no_middle_anchor c
JOIN sampled_anchors s ON s.anchor_id = c.anchor_id
ORDER BY c.anchor_id, c.straggler_works_count DESC


### Cell 4: Create audit log


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_merge_log_no_middle_anchor AS
SELECT
  anchor_id              AS winner_author_id,
  anchor_orcid           AS winner_orcid,
  straggler_id           AS loser_author_id,
  straggler_orcid        AS loser_orcid,
  straggler_full_name    AS loser_full_name,
  straggler_works_count  AS loser_works_count,
  straggler_cited_by_count AS loser_cited_by_count,
  straggler_created_date AS loser_created_date,
  p_first, p_middle, p_last,
  inst_overlap,
  n_shared_coauthors,
  current_timestamp()    AS merge_timestamp
FROM openalex.authors.merge_candidates_no_middle_anchor


### Cell 5: Snapshot affected work_ids (capture before MERGE rewrites author_id)


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.authors.author_merge_affected_works_no_middle_anchor AS
SELECT DISTINCT wa.work_id
FROM openalex.works.work_authors wa
JOIN openalex.authors.author_merge_log_no_middle_anchor log
  ON wa.author_id = log.loser_author_id


### Cell 6: Execute the merge — rewrite work_authors.author_id from losers to winners


In [ ]:
%sql
MERGE INTO openalex.works.work_authors AS target
USING (
  SELECT loser_author_id, winner_author_id
  FROM openalex.authors.author_merge_log_no_middle_anchor
) AS source
ON target.author_id = source.loser_author_id
WHEN MATCHED THEN
  UPDATE SET
    target.author_id = source.winner_author_id,
    target.updated_at = current_timestamp()


### Cell 7: Queue affected works for reprocessing by UpdateWorkAuthorships

*No DELETE step. Phases 1 / 2 / 3b / 3c all use the MERGE-then-soft-zero-via-CreateAuthors approach; this notebook follows that pattern.*


In [ ]:
%sql
INSERT INTO openalex.institutions.curated_work_ids_pending_sync
SELECT work_id, NULL AS curated_ras, current_timestamp() AS added_datetime
FROM openalex.authors.author_merge_affected_works_no_middle_anchor


### Cell 8: Post-merge verification — losers should drain to 0 works after CreateAuthors


In [ ]:
%sql
WITH log AS (
  SELECT loser_author_id, loser_works_count
  FROM openalex.authors.author_merge_log_no_middle_anchor
),
loser_state AS (
  SELECT l.loser_author_id, l.loser_works_count,
         a.works_count AS current_works_count
  FROM log l
  JOIN openalex.authors.openalex_authors a ON a.id = l.loser_author_id
)
SELECT
  COUNT(*)                                                       AS total_losers,
  COUNT(CASE WHEN current_works_count = 0 THEN 1 END)            AS losers_drained_to_zero,
  COUNT(CASE WHEN current_works_count > 0 THEN 1 END)            AS losers_still_nonzero,
  PERCENTILE_APPROX(current_works_count, ARRAY(0.5, 0.95, 0.99)) AS nonzero_pctiles,
  MAX(current_works_count)                                       AS max_remaining_works
FROM loser_state
